General naming principles _Value is the end unit used in the cost test

All pieces need to be for the full measure life

Real discount rate is the time value of money and needs to be multipled against the provided real dollar avoided cost inputs
once the real value of money is calculated those inputs are used for each measure's lifetime
Note this is just for the first year screening process
the Measure_Counts_to_Values.ipynb will have to be able to pull the cost of each measure starting at different times (this process happens after the adoption model)

In [627]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [628]:
#Column Cleaning Function
#change column names to be lowercase and replace spaces with underscores
# change $ to usd
def clean_column_names(df):
    df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('$', 'usd').str.replace('/', 'per').str.replace('&', 'and').str.replace('.', '')
    return df

In [629]:
#Read in Assembled Measures
df = pd.read_csv("input/assembled_measures.csv")
clean_column_names(df)
df['incremental_cost'] = df['incremental_installed_cost_(usd)']
df['measure_electric_energy_savings'] = df['energy_impact_1']

#needs to be pulled in to assembled measures
#########################
#### Feature Request ####
#########################

# to calculate this value for each building type divided the flh by the flh by 8760 # hours in a year
#new column!
df['demand_ratio'] = 1/8760
df['demand_ratio'] = df['demand_ratio'].astype(float)

#################################################
### Inputs From Measure Table and Adjustments ###
#################################################
# each of these are for a single installation of this measure (one widget)
# all costs and benefits are for just one year but the tests are run on Whole measures lives

df['measure_incremental_cost'] = df['incremental_cost'] #raw input from measure table


df["deferred_replacement_credit_value"] = 0 # pulled in from assembled measures 
#MD always 0 
# RET ->((df[baseline_measure_cost(with install)] *(2/3) ) / ((1 + real_discount_rate) ** (1/3 * df['EUL']))
# the baseline 2/3 or the RUL of the baseline measure net present valued to todays dollars (NOte always the same RUL because we assume this is the average of retrofitted measure for adoption model)
# Needs to be NPV of future avoided replacement costs due to measure extending equipment life
"""Deferred Replacement Credit: For an early-retirement retrofit measure, a credit for deferring the
 replacement cycle of the existing equipment, due to early retirement of existing equipment. See Data Only
   for Early-retirement Retrofit Measures, p. 27."""

""" there is a benefit in replacing an older equipment with an EUL less than the new equipment - that is what this is capturing""" # But this is Not that!!!!!!
### Need to fix input data to have fuel source
df['measure_natural_gas_savings'] = 10
df['measure_fuel_oil_savings'] = 10
df['measure_propane_savings'] = 10
df['measure_gasoline_savings'] = 1
df['measure_diesel_savings'] = 1
df['measure_water_savings'] = 1 #df['water_savings_(gallons)']


In [630]:
#Read in avoided costs data
avoided_costs = pd.read_excel("input/11_Avoided_Cost.xlsx")
clean_column_names(avoided_costs)

#########################
#### Feature Request ####
#########################
# Requested feature from Matt - Have real discount rate change based on Cost test used
# so now we need to do all of these calculations for each and every cost test
# now we need to convert all avoided costs (that are in real dollars of the start year) to discounted dollars due to the time value of money
# Money today is worth more than the same amount of money in the future due to its potential earning capacity
real_discount_rate = 0.03
# this value will need to be changeable based on test run
# I think we do this calculation for each test and just label the column with the test name that it will need to be used for

##############################
#### Applying Discounting ####
##############################
# Present Value = Future Value / (1 + real_discount_rate)^(year - base_year)
# Get list of columns that contain dollar values (looking for 'usd' in column name)
dollar_columns = [col for col in avoided_costs.columns if 'usd' in col.lower()]
avoided_costs['base_year'] = avoided_costs['year'].min()
# Apply discount formula to each dollar column
for col in dollar_columns:
    avoided_costs[col] = avoided_costs[col] / ((1 + real_discount_rate) ** (avoided_costs['year'] - avoided_costs['base_year']))

avoided_costs

PermissionError: [Errno 13] Permission denied: 'input/11_Avoided_Cost.xlsx'

In [ ]:
# read in loadshapes data
loadshapes = pd.read_excel("input/12_loadshapes.xlsx")
clean_column_names(loadshapes)
loadshapes
# column_names = avoided_costs.columns.tolist()
# print(column_names)

,condition_name,competition_group,subgroup,summer_on-peak,summer_off-peak,winter_on-peak,winter_off-peak,shoulder_on-peak,shoulder_off-peak,summer_gener_capacity,winter_gener_capacity,summer_tandd,winter_tandd
0,furnace_fuel_oil_existing_residential,heating_cooling,oil_furnace,0.007878,0.018346,0.281812,0.368809,0.129864,0.193292,0.000000,0.381007,0.000000,0.381007
1,furnace_natural_gas_baseline_residential,heating_cooling,gas_furnace,0.007878,0.018346,0.281812,0.368809,0.129864,0.193292,0.000000,0.381007,0.000000,0.381007
2,furnace_natural_gas_efficient_residential,heating_cooling,gas_furnace,0.007878,0.018346,0.281812,0.368809,0.129864,0.193292,0.000000,0.381007,0.000000,0.381007
3,room_ac_electricity_baseline_residential,heating_cooling,room_ac,0.490621,0.371577,0.020994,0.026759,0.048400,0.041650,0.372861,0.000000,0.372861,0.000000
4,room_ac_electricity_efficient_residential,heating_cooling,room_ac,0.490621,0.371577,0.020994,0.026759,0.048400,0.041650,0.372861,0.000000,0.372861,0.000000
5,air_conditioner_electricity_baseline_residential,heating_cooling,central_ac,0.490621,0.371577,0.020994,0.026759,0.048400,0.041650,0.372861,0.000000,0.372861,0.000000
6,air_conditioner_electricity_efficient_residential,heating_cooling,central_ac,0.490621,0.371577,0.020994,0.026759,0.048400,0.041650,0.372861,0.000000,0.372861,0.000000
7,cchp_electricity_efficient_residential,heating_cooling,cchp,0.490621,0.371577,0.020994,0.026759,0.048400,0.041650,0.372861,0.000000,0.372861,0.000000
8,refrigerator_electricity_existing_residential,refrigeration,full_size,0.220863,0.237399,0.095015,0.110339,0.164724,0.171661,1.186104,0.888264,1.186104,0.888264
9,refrigerator_electricity_baseline_residential,refrigeration,full_size,0.220863,0.237399,0.095015,0.110339,0.164724,0.171661,1.186104,0.888264,1.186104,0.888264


In [ ]:
# this function allows us to grab certain groups of columns needed in the analysis

def weighted_column_sum(df, weights_row, columns=None, fill_value=0.0):
    """
    df: DataFrame with values to weight (e.g. loadshapes)
    weights_row: Series-like with weights indexed by column name (e.g. avoided_costs.loc[0])
    columns: optional list of columns to use; if None, intersection of df.columns and weights_row.index
    """
    if columns is None:
        columns = df.columns.intersection(weights_row.index)
    w = pd.Series(weights_row).reindex(columns).astype(float).fillna(fill_value)
    return df[columns].fillna(fill_value).dot(w)

#Excel QC complete

In [ ]:
# Line Losses
line_losses = pd.read_excel("input/15_Line_Losses.xlsx")
clean_column_names(line_losses)
line_losses

,sectors,summer_on-peak,summer_off-peak,winter_on-peak,winter_off-peak,shoulder_on-peak,shoulder_off-peak,summer_gener_capacity,winter_gener_capacity,summer_tandd,winter_tandd
0,res,0.0943,0.0943,0.0943,0.0943,0.0943,0.0943,0.0943,0.0943,0.0943,0.0943
1,com,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790
2,ind,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790,0.0790


In [ ]:
###################################################################
## Electric Energy Savings Calculation with line-loss adjustment ##
###################################################################

# Find columns common to all three tables (avoided_costs, loadshapes, line_losses)
avoided_costs_electric_use = avoided_costs.iloc[:, :7]
#edit columns to drop some text so we can use the function

avoided_costs_electric_use.columns = [col.replace('_usdperkwh', '') for col in avoided_costs_electric_use.columns]

common = loadshapes.columns.intersection(avoided_costs_electric_use.columns).intersection(line_losses.columns)

# Pre-compute the loadshape vector (period weights) - assume single row of loadshapes describes the profile
loadshape_vector = loadshapes[common].fillna(0).iloc[0].astype(float)

# Real discount rate for NPV calculation
real_discount_rate = 0.03

# Helper to map df sector text to loss table key
def sector_key_from_sector_text(sector_text):
    s = str(sector_text).lower() if sector_text is not None else ''
    if 'res' in s or 'resident' in s or 'house' in s:
        return 'res'
    if 'com' in s or 'commercial' in s:
        return 'com'
    if 'ind' in s or 'industrial' in s:
        return 'ind'
    # fallback: try to match exact short codes if present
    if sector_text in ['res','com','ind']:
        return sector_text
    return None

# NPV calculation: for each row in df, compute NPV of electric energy savings across measure lifetime
def compute_electric_energy_npv(row):
    """
    Compute NPV of electric energy savings across measure lifetime using sector-specific line losses.
    """
    measure_life = int(row.get('measure_life_(yrs)', 1))
    elec_savings = row.get('measure_electric_energy_savings', 0)
    
    if pd.isna(elec_savings) or elec_savings == 0:
        return 0.0

    # determine sector key for selecting line_losses row
    sector_k = sector_key_from_sector_text(row.get('sector', None))

    npv_total = 0.0
    base_year = avoided_costs_electric_use['year'].min()

    # For each year in the measure life
    for year_offset in range(1, measure_life + 1):
        target_year = base_year + year_offset

        # Get avoided costs for this year (or use last available if beyond data)
        if target_year in avoided_costs_electric_use['year'].values:
            year_costs = avoided_costs_electric_use[avoided_costs_electric_use['year'] == target_year][common].iloc[0].astype(float).fillna(0)
        else:
            # Use last available year if beyond data range
            year_costs = avoided_costs_electric_use[common].iloc[-1].astype(float).fillna(0)

        # Select line_losses row for this sector (try likely column names 'sector' or 'sectors')
        loss_row = None
        for lname in ['sector', 'sectors']:
            if lname in line_losses.columns:
                try:
                    mask = line_losses[lname].astype(str).str.lower().str.contains(str(sector_k))
                except Exception:
                    mask = pd.Series([False] * len(line_losses), index=line_losses.index)
                if mask.any():
                    loss_row = line_losses.loc[mask, common].iloc[0].astype(float).fillna(0)
                    break
        # Fallback to first row if no sector-matched row found # this could cause issues
        if loss_row is None:
            loss_row = line_losses.loc[0, common].astype(float).fillna(0)

        # Compute weighted sum for this year (loadshape * (1 - line_loss) * avoided_cost)
        year_value = (loadshape_vector * (1 + loss_row)).dot(year_costs)

        # Apply discount factor and accumulate NPV
        discount_factor = 1.0 / ((1 + real_discount_rate) ** year_offset)
        npv_total += elec_savings * year_value * discount_factor

    return npv_total

# Apply NPV calculation to each row
df['electric_energy_savings_value'] = df.apply(compute_electric_energy_npv, axis=1)

df
#excel QC complete

# OLD CODE (archived - using first row only, simple multiplication):
# weights = avoided_costs_electric_use.loc[0, common].astype(float).fillna(0)
# losses = line_losses.loc[0, common].astype(float).fillna(0) # in future need to add ability to apply the correct sector
# adjusted_weights = weights * (1 - losses)
# period_value = loadshapes[common].fillna(0).dot(adjusted_weights)
# df['electric_energy_savings_value'] = period_value * df['measure_electric_energy_savings'] * df['measure_life_(yrs)']

,measure_name,sector,program,electric_utility,gas_utility,market,baseline_condition,efficient_condition,building_type,electric_end_use,...,demand_ratio,measure_incremental_cost,deferred_replacement_credit_value,measure_natural_gas_savings,measure_fuel_oil_savings,measure_propane_savings,measure_gasoline_savings,measure_diesel_savings,measure_water_savings,electric_energy_savings_value
0,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,0.000114,650,0,10,10,10,1,1,1,282.221049
1,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,...,0.000114,650,0,10,10,10,1,1,1,470.368415
2,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,...,0.000114,650,0,10,10,10,1,1,1,470.368415
3,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family,unknown,...,0.000114,650,0,10,10,10,1,1,1,282.221049
4,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,0.000114,450,0,10,10,10,1,1,1,181.427817
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,0.000114,77,0,10,10,10,1,1,1,0.000000
300,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,RENO,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,0.000114,77,0,10,10,10,1,1,1,0.000000
301,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,NC,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,0.000114,-2050,0,10,10,10,1,1,1,80.450160
302,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,0.000114,-2050,0,10,10,10,1,1,1,80.450160


In [ ]:
# ###################################################################
# ## Electric Energy DEMAND Calculation with line-loss adjustment ##
# ###################################################################
# # Make global inputs for established energy periods 
# # Now condition-aware: match df['efficient_condition'] to loadshapes['condition_name']

# # Helper to get loadshape values for a given condition
# def get_loadshape_values(condition_name):
#     """
#     Fetch loadshape values for a given condition_name.
#     Returns a dict with the period-specific loadshape values.
#     If condition_name not found, fallback to first row.
#     """
#     if 'condition_name' in loadshapes.columns:
#         matching = loadshapes[loadshapes['condition_name'] == condition_name]
#         if not matching.empty:
#             row = matching.iloc[0]
#         else:
#             # fallback to first row if condition not found will need to change
#             row = loadshapes.iloc[0]
#     else:
#         # If no condition_name column, use first row
#         row = loadshapes.iloc[0]
    
#     return {
#         'summer_gener_capacity': row.get('summer_gener_capacity', 0),
#         'summer_tandd': row.get('summer_tandd', 0),
#         'winter_gener_capacity': row.get('winter_gener_capacity', 0),
#         'winter_tandd': row.get('winter_tandd', 0)
#     }

# # Function to compute demand savings per row with condition-based loadshapes
# def compute_electric_demand_savings(row):
#     """
#     Compute electric demand savings for this row using:
#     - condition_name from row to select appropriate loadshape values
#     - avoided costs (first row for now, could be extended to year-by-year)
#     - line losses (first row for now, could be extended to sector-aware)
#     """
#     demand_ratio = row.get('demand_ratio', 1.0 / 8760)
#     elec_energy_savings = row.get('measure_electric_energy_savings', 0)
    
#     if pd.isna(elec_energy_savings) or elec_energy_savings == 0:
#         return 0.0
    
#     # Get loadshape values for this row's condition
#     condition = row.get('efficient_condition', None)
#     ls_vals = get_loadshape_values(condition)
    
#     # Compute demand savings using condition-specific loadshapes
#     demand_savings = demand_ratio * elec_energy_savings * (
#         (avoided_costs.loc[0, 'summer_gener_capacity_usdperkw-yr'] * (1 + line_losses.loc[0, 'summer_gener_capacity']) * ls_vals['summer_gener_capacity'])
#         + (avoided_costs.loc[0, 'summer_td_usdperkw-yr'] * (1 + line_losses.loc[0, 'summer_tandd']) * ls_vals['summer_tandd'])
#         + (avoided_costs.loc[0, 'winter_gener_capacity_usdperkw-yr'] * (1 + line_losses.loc[0, 'winter_gener_capacity']) * ls_vals['winter_gener_capacity'])
#         + (avoided_costs.loc[0, 'winter_td_usdperkw-yr'] * (1 + line_losses.loc[0, 'winter_tandd']) * ls_vals['winter_tandd'])
#     )
#     return demand_savings

# # Apply per-row calculation
# df['electric_demand_savings_value'] = df.apply(compute_electric_demand_savings, axis=1)

# df
# # (Energy saved times the kw/kwh ratio (this needs to be added to the measure output) * summer_gen_capacity(CF)from loadshapes * summer_gen_capacity_cost from Avoided cost) same process for winter and T&D sets? then just add together
# #line losses all modeled impacts are at the meter all avoided costs are at generation so need to adjust energy or demand impacts from meter to at gen (so this is just multiply)
# # i think we need to add back the line losses not remove them?
# # Excel QC needed
# # outline of proper yearly avoided cost also needed

In [ ]:
###################################################################
# Electric Demand NPV Calculation (standalone cell)
# This cell contains the helper and per-row NPV function for demand-related avoided costs
###################################################################

# Helper to get loadshape values for a given condition
# (If you already have this defined elsewhere in the notebook, this re-definition is safe and will override.)
def get_loadshape_values(condition_name):
    """
    Fetch loadshape values for a given condition_name.
    Returns a dict with the period-specific loadshape values.
    If condition_name not found, fallback to first row.
    """
    if 'condition_name' in loadshapes.columns:
        matching = loadshapes[loadshapes['condition_name'] == condition_name]
        if not matching.empty:
            row = matching.iloc[0]
        else:
            # fallback to first row if condition not found
            row = loadshapes.iloc[0]
    else:
        # If no condition_name column, use first row
        row = loadshapes.iloc[0]
    
    return {
        'summer_gener_capacity': row.get('summer_gener_capacity', 0),
        'summer_tandd': row.get('summer_tandd', 0),
        'winter_gener_capacity': row.get('winter_gener_capacity', 0),
        'winter_tandd': row.get('winter_tandd', 0)
    }

# Real discount rate for NPV calculation
real_discount_rate = 0.03

# Function to compute lifetime NPV of demand savings per row with condition-based loadshapes
def compute_electric_demand_npv(row):
    """
    Compute lifetime NPV of electric demand savings for this row using:
    - condition_name from row to select appropriate loadshape values
    - year-by-year avoided costs for demand (summer and winter capacity, T&D)
    - line losses (first row used here; consider making sector-aware if needed)
    - Full measure lifetime with 3% real discount rate applied
    """
    measure_life = int(row.get('measure_life_(yrs)', 1))
    demand_ratio = row.get('demand_ratio', 1.0 / 8760)
    elec_energy_savings = row.get('measure_electric_energy_savings', 0)
    
    if pd.isna(elec_energy_savings) or elec_energy_savings == 0:
        return 0.0
    
    # Get loadshape values for this row's condition
    condition = row.get('efficient_condition', None)
    ls_vals = get_loadshape_values(condition)
    
    # Get line loss values (first row for now)
    summer_gen_loss = line_losses.loc[0, 'summer_gener_capacity'] if 'summer_gener_capacity' in line_losses.columns else 0
    summer_td_loss = line_losses.loc[0, 'summer_tandd'] if 'summer_tandd' in line_losses.columns else 0
    winter_gen_loss = line_losses.loc[0, 'winter_gener_capacity'] if 'winter_gener_capacity' in line_losses.columns else 0
    winter_td_loss = line_losses.loc[0, 'winter_tandd'] if 'winter_tandd' in line_losses.columns else 0
    
    npv_total = 0.0
    base_year = avoided_costs['year'].min()
    
    # For each year in the measure life
    for year_offset in range(1, measure_life + 1):
        target_year = base_year + year_offset
        
        # Get avoided costs for this year (or use last available if beyond data)
        if target_year in avoided_costs['year'].values:
            year_costs = avoided_costs[avoided_costs['year'] == target_year].iloc[0]
        else:
            # Use last available year if beyond data range
            year_costs = avoided_costs.iloc[-1]
        
        # Extract demand-related avoided cost values for this year
        summer_gen_cost = year_costs.get('summer_gener_capacity_usdperkw-yr', 0)
        summer_td_cost = year_costs.get('summer_td_usdperkw-yr', 0)
        winter_gen_cost = year_costs.get('winter_gener_capacity_usdperkw-yr', 0)
        winter_td_cost = year_costs.get('winter_td_usdperkw-yr', 0)
        
        # Compute weighted demand value for this year using condition-specific loadshapes
        year_demand_value = (
            (summer_gen_cost * (1 + summer_gen_loss) * ls_vals['summer_gener_capacity'])
            + (summer_td_cost * (1 + summer_td_loss) * ls_vals['summer_tandd'])
            + (winter_gen_cost * (1 + winter_gen_loss) * ls_vals['winter_gener_capacity'])
            + (winter_td_cost * (1 + winter_td_loss) * ls_vals['winter_tandd'])
        )
        
        # Apply discount factor and accumulate NPV
        discount_factor = 1.0 / ((1 + real_discount_rate) ** year_offset)
        npv_total += demand_ratio * elec_energy_savings * year_demand_value * discount_factor
    
    return npv_total

# Apply per-row calculation (creates/overwrites df['electric_demand_savings_value'])
df['electric_demand_savings_value'] = df.apply(compute_electric_demand_npv, axis=1)

# Quick check (print a small sample)
print(df[['efficient_condition','measure_life_(yrs)','measure_electric_energy_savings','electric_demand_savings_value']].head())

                              efficient_condition  measure_life_(yrs)  \
0  refrigerator_electricity_efficient_residential                  10   
1  refrigerator_electricity_efficient_residential                  10   
2  refrigerator_electricity_efficient_residential                  10   
3  refrigerator_electricity_efficient_residential                  10   
4  refrigerator_electricity_efficient_residential                  10   

   measure_electric_energy_savings  electric_demand_savings_value  
0                       700.000000                      62.272771  
1                      1166.666667                     103.787952  
2                      1166.666667                     103.787952  
3                       700.000000                      62.272771  
4                       450.000000                      40.032496  


In [ ]:
###########################################################
## Natural Gas Savings (Full Measure Lifetime)         ##
###########################################################
# Calculate the lifetime savings over the full measure lifetime
# The avoided costs are already discounted, so we just sum them across the measure life

def calculate_lifetime_savings(measure_savings, avoided_cost_series, measure_life):
    """
    Calculate the NPV of savings over the measure's full lifetime
    
    measure_savings: annual savings (scalar)
    avoided_cost_series: series of already-discounted avoided costs indexed by year
    measure_life: number of years the measure lasts
    """
    # Sum the discounted avoided costs across all years of measure life
    lifetime_savings = 0
    for year_offset in range(1, measure_life + 1):
        year = avoided_cost_series.index.min() + year_offset
        if year in avoided_cost_series.index:
            annual_cost = avoided_cost_series[year]
        else:
            # If year is beyond available data, use the last available year
            annual_cost = avoided_cost_series.iloc[-1]
        
        lifetime_savings += measure_savings * annual_cost
    
    return lifetime_savings

# Apply to natural gas savings for each row
df['natural_gas_savings_value'] = df.apply(
    lambda row: calculate_lifetime_savings(
        row['measure_natural_gas_savings'],
        avoided_costs.set_index('year')['natural_gas_usdpermmbtu'],
        int(row['measure_life_(yrs)'])
    ),
    axis=1
)
df 
# Excel QC Complete

,measure_name,sector,program,electric_utility,gas_utility,market,baseline_condition,efficient_condition,building_type,electric_end_use,...,deferred_replacement_credit_value,measure_natural_gas_savings,measure_fuel_oil_savings,measure_propane_savings,measure_gasoline_savings,measure_diesel_savings,measure_water_savings,electric_energy_savings_value,electric_demand_savings_value,natural_gas_savings_value
0,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,0,10,10,10,1,1,1,282.221049,62.272771,417.918450
1,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,...,0,10,10,10,1,1,1,470.368415,103.787952,417.918450
2,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,...,0,10,10,10,1,1,1,470.368415,103.787952,417.918450
3,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family,unknown,...,0,10,10,10,1,1,1,282.221049,62.272771,417.918450
4,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,0,10,10,10,1,1,1,181.427817,40.032496,417.918450
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,0,10,10,10,1,1,1,0.000000,0.000000,956.828411
300,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,RENO,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,0,10,10,10,1,1,1,0.000000,0.000000,956.828411
301,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,NC,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,0,10,10,10,1,1,1,80.450160,0.000000,956.828411
302,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,0,10,10,10,1,1,1,80.450160,0.000000,956.828411


In [ ]:
# df['other_fuel_savings'] = #raw input from measure table * other fuel avoided cost from avoided costs table
#################################
#### Other fuel Calculations ####
#################################
# These calculations are performed based on sector matching
# Sector prefixes: res = residential, com = commercial, ind = industrial
# Fuel types: fuel_oil, propane, diesel_transportation, gasoline_transportation

# Define fuel type mappings: (fuel_type, measure_column, avoided_cost_columns_by_sector)
fuel_configs = [
    ('fuel_oil', 'measure_fuel_oil_savings', {
        'res': 'res_fuel_oil_usdpermmbtu',
        'com': 'com_fuel_oil_usdpermmbtu',
        'ind': 'ind_fuel_oil_usdpermmbtu'
    }),
    ('propane', 'measure_propane_savings', {
        'res': 'res_propane_usdpermmbtu',
        'com': 'com_propane_usdpermmbtu',
        'ind': 'ind_propane_usdpermmbtu'
    }),
    # I just did this to make the loop more complete # ther is probably a better way?
    ('diesel_transportation', 'measure_diesel_savings', {
        'res': 'diesel_transportation_usdpermmbtu',
        'com': 'diesel_transportation_usdpermmbtu',
        'ind': 'diesel_transportation_usdpermmbtu'
    }),
    ('gasoline_transportation', 'measure_gasoline_savings', {
        'res': 'gasoline_transportation_usdpermmbtu',
        'com': 'gasoline_transportation_usdpermmbtu',
        'ind': 'gasoline_transportation_usdpermmbtu'
    })
]

# Process each fuel type
for fuel_type, measure_col, sector_columns in fuel_configs:
    output_col = f'{fuel_type}_savings_value'
    
    def calculate_sector_fuel_savings(row):
        """Calculate lifetime savings based on sector and fuel type"""
        # Determine sector from the 'sector' column (case-insensitive)
        sector = None
        sector_lower = str(row.get('sector', '')).lower()
        
        if 'res' in sector_lower:
            sector = 'res'
        elif 'com' in sector_lower:
            sector = 'com'
        elif 'ind' in sector_lower:
            sector = 'ind'
        else:
            return 0  # No matching sector
        
        # Get the appropriate avoided cost column for this sector
        avoided_cost_col = sector_columns.get(sector)
        
        # Check if the column exists in avoided_costs
        if avoided_cost_col not in avoided_costs.columns:
            return 0
        
        # Check if measure_col exists in the row and get measure savings
        if measure_col not in row.index or pd.isna(row[measure_col]):
            return 0
        
        measure_savings = row[measure_col]
        
        # Calculate lifetime savings using the existing function
        try:
            return calculate_lifetime_savings(
                measure_savings,
                avoided_costs.set_index('year')[avoided_cost_col],
                int(row['measure_life_(yrs)'])
            )
        except:
            return 0
    
    # Apply the calculation for this fuel type
    df[output_col] = df.apply(calculate_sector_fuel_savings, axis=1)

# Excel QC Complete
df

,measure_name,sector,program,electric_utility,gas_utility,market,baseline_condition,efficient_condition,building_type,electric_end_use,...,measure_gasoline_savings,measure_diesel_savings,measure_water_savings,electric_energy_savings_value,electric_demand_savings_value,natural_gas_savings_value,fuel_oil_savings_value,propane_savings_value,diesel_transportation_savings_value,gasoline_transportation_savings_value
0,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,1,1,1,282.221049,62.272771,417.918450,2275.807546,2306.476785,249.868869,221.840539
1,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,...,1,1,1,470.368415,103.787952,417.918450,2275.807546,2306.476785,249.868869,221.840539
2,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,...,1,1,1,470.368415,103.787952,417.918450,2275.807546,2306.476785,249.868869,221.840539
3,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family,unknown,...,1,1,1,282.221049,62.272771,417.918450,2275.807546,2306.476785,249.868869,221.840539
4,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,1,1,1,181.427817,40.032496,417.918450,2275.807546,2306.476785,249.868869,221.840539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,1,1,1,0.000000,0.000000,956.828411,5591.202670,5832.277807,604.764082,489.446540
300,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,RENO,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,1,1,1,0.000000,0.000000,956.828411,5591.202670,5832.277807,604.764082,489.446540
301,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,NC,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,1,1,1,80.450160,0.000000,956.828411,5591.202670,5832.277807,604.764082,489.446540
302,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,1,1,1,80.450160,0.000000,956.828411,5591.202670,5832.277807,604.764082,489.446540


In [ ]:
df['other_fuel_savings_value'] = df['fuel_oil_savings_value'] + df['propane_savings_value'] + df['diesel_transportation_savings_value'] + df['gasoline_transportation_savings_value']


In [ ]:
# oandm savings are already in the dollar value for a single unit
# this value may be characterized as just the difference in oandm costs between the baseline and measure condition
# NPV of O&M costs over measure lifetime
# Each year's O&M cost is discounted back to present value
real_discount_rate = 0.03
df['annual_oandm_cost'] = 10
df['o_and_m_savings_value'] = df.apply(
    lambda row: sum(
        row['annual_oandm_cost'] / ((1 + real_discount_rate) ** year)
        for year in range(1, int(row['measure_life_(yrs)']) + 1)
    ),
    axis=1
)
df

,measure_name,sector,program,electric_utility,gas_utility,market,baseline_condition,efficient_condition,building_type,electric_end_use,...,measure_water_savings,electric_energy_savings_value,electric_demand_savings_value,natural_gas_savings_value,fuel_oil_savings_value,propane_savings_value,diesel_transportation_savings_value,gasoline_transportation_savings_value,other_fuel_savings_value,o_and_m_savings_value
0,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,1,282.221049,62.272771,417.918450,2275.807546,2306.476785,249.868869,221.840539,5053.993739,85.302028
1,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,...,1,470.368415,103.787952,417.918450,2275.807546,2306.476785,249.868869,221.840539,5053.993739,85.302028
2,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,...,1,470.368415,103.787952,417.918450,2275.807546,2306.476785,249.868869,221.840539,5053.993739,85.302028
3,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family,unknown,...,1,282.221049,62.272771,417.918450,2275.807546,2306.476785,249.868869,221.840539,5053.993739,85.302028
4,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,1,181.427817,40.032496,417.918450,2275.807546,2306.476785,249.868869,221.840539,5053.993739,85.302028
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,1,0.000000,0.000000,956.828411,5591.202670,5832.277807,604.764082,489.446540,12517.691099,196.004413
300,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,RENO,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,1,0.000000,0.000000,956.828411,5591.202670,5832.277807,604.764082,489.446540,12517.691099,196.004413
301,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,NC,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,1,80.450160,0.000000,956.828411,5591.202670,5832.277807,604.764082,489.446540,12517.691099,196.004413
302,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,1,80.450160,0.000000,956.828411,5591.202670,5832.277807,604.764082,489.446540,12517.691099,196.004413


In [ ]:
#df['water_savings_value'] = df['measure_water_savings'] *avoided_costs.loc[0,'water_usdpergallon']
#raw input from measure table * water avoided cost from avoided costs table

df['water_savings_value'] = df.apply(
    lambda row: calculate_lifetime_savings(
        row['measure_water_savings'],
        avoided_costs.set_index('year')['water_usdpergallon'],
        int(row['measure_life_(yrs)'])
    ),
    axis=1
)
df 

,measure_name,sector,program,electric_utility,gas_utility,market,baseline_condition,efficient_condition,building_type,electric_end_use,...,electric_energy_savings_value,electric_demand_savings_value,natural_gas_savings_value,fuel_oil_savings_value,propane_savings_value,diesel_transportation_savings_value,gasoline_transportation_savings_value,other_fuel_savings_value,o_and_m_savings_value,water_savings_value
0,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,282.221049,62.272771,417.918450,2275.807546,2306.476785,249.868869,221.840539,5053.993739,85.302028,0.145013
1,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,...,470.368415,103.787952,417.918450,2275.807546,2306.476785,249.868869,221.840539,5053.993739,85.302028,0.145013
2,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,...,470.368415,103.787952,417.918450,2275.807546,2306.476785,249.868869,221.840539,5053.993739,85.302028,0.145013
3,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family,unknown,...,282.221049,62.272771,417.918450,2275.807546,2306.476785,249.868869,221.840539,5053.993739,85.302028,0.145013
4,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,181.427817,40.032496,417.918450,2275.807546,2306.476785,249.868869,221.840539,5053.993739,85.302028,0.145013
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,0.000000,0.000000,956.828411,5591.202670,5832.277807,604.764082,489.446540,12517.691099,196.004413,0.333208
300,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,RENO,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,0.000000,0.000000,956.828411,5591.202670,5832.277807,604.764082,489.446540,12517.691099,196.004413,0.333208
301,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,NC,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,80.450160,0.000000,956.828411,5591.202670,5832.277807,604.764082,489.446540,12517.691099,196.004413,0.333208
302,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,80.450160,0.000000,956.828411,5591.202670,5832.277807,604.764082,489.446540,12517.691099,196.004413,0.333208


In [ ]:
### Incentives
# Incentive values are a fraction of the measure_incremental_cost depending on program, building type, end_use and install type 
# Input table 13_incentives 
# For now we just do incentives at the measure level and will have the percent tables something in excel other could use
incentives = pd.read_excel("input/13_Incentives.xlsx")
clean_column_names(incentives)
# will need to make sure there is no double counting here
# because are initial counts will be specific to the combination (test_utility_gas & test utility electric) we should
# have a electric and gas utility for each and the incentives need to know this and not add up to more than the incremental cost

# The market characterization file will add in the information on who is providing incentives
# we need to add a loop here that makes sure that we do not exceed 100% of the incremental cost and that the approprate utitlity incentives are being pulled in
# for instance if only electric utility is providing incentive then only electric utility percentage should be used (with the right name)
# might need to do deferred replacement in here

# upgrades - CUrrently this requires a none in the input data. which I think makes sense
# lots of checks that should probably be integrated in the pipeline but arent necessary for the code to work
# --- Incentives: match by measure_name and utility and compute dollar incentives ---

# Normalize / detect key columns (some files may have slightly different names)
# Required: 'measure_name' should exist in incentives (if not, try common variants)
if 'measure_name' not in incentives.columns:
    for alt in ['measure', 'measureid', 'measure_id', 'measure_name_']:
        if alt in incentives.columns:
            incentives = incentives.rename(columns={alt: 'measure_name'})
            break

# Utility column may or may not exist; if it does, we'll match per-utility; otherwise fallback to per-measure values
utility_col_name = None
for c in ['utility', 'utility_name', 'program_utility', 'provider']:
    if c in incentives.columns:
        utility_col_name = c
        break

# Find the likely percentage columns (tolerant to small name differences)
def find_col(df, must_have):
    for col in df.columns:
        low = col.lower()
        if all(part in low for part in must_have):
            return col
    return None

electric_pct_col = find_col(incentives, ['electric', 'incent', 'percent']) or find_col(incentives, ['electric', 'incentive'])
gas_pct_col = find_col(incentives, ['natural', 'gas', 'incent', 'percent']) or find_col(incentives, ['gas', 'incent'])
nonutility_pct_col = find_col(incentives, ['non', 'util', 'incent', 'percent']) or find_col(incentives, ['nonutility', 'incent'])

# As a defensive fallback, try obvious exact names used previously
if electric_pct_col is None and 'electric_incentive_percentage' in incentives.columns:
    electric_pct_col = 'electric_incentive_percentage'
if gas_pct_col is None and 'natural_gas_incentive_percentage' in incentives.columns:
    gas_pct_col = 'natural_gas_incentive_percentage'
if nonutility_pct_col is None and 'nonutility_incentive_percentage' in incentives.columns:
    nonutility_pct_col = 'nonutility_incentive_percentage'

# If any of those are still missing, set to None and we'll treat missing as 0
# Helper to get percentage for a given measure_name and utility (with fallbacks)
def get_incentive_pct(measure_name, utility_name, pct_col):
    if pct_col is None:
        return 0.0
    # try exact measure+utility if utility column exists
    if utility_col_name is not None and utility_name is not None:
        match = incentives[(incentives['measure_name'] == measure_name) & (incentives[utility_col_name] == utility_name)]
        if not match.empty and pd.notna(match.iloc[0].get(pct_col)):
            return float(match.iloc[0][pct_col])
    # next fallback: any row with the measure_name (ignore utility)
    match2 = incentives[incentives['measure_name'] == measure_name]
    if not match2.empty:
        # prefer a row where the pct_col is not null
        non_null = match2[match2[pct_col].notna()]
        if not non_null.empty:
            return float(non_null.iloc[0][pct_col])
        # otherwise take first row even if NaN (will become 0)
        val = match2.iloc[0].get(pct_col)
        return float(val) if pd.notna(val) else 0.0
    # last fallback: try to use a global/default in the incentives table (first row)
    if pct_col in incentives.columns and not incentives.empty:
        val = incentives.iloc[0].get(pct_col)
        return float(val) if pd.notna(val) else 0.0
    return 0.0

# Determine what column in the main df holds the gas utility name
gas_utility_col = None
for c in ['gas_utility', 'natural_gas_utility', 'gasutility', 'gas_provider']:
    if c in df.columns:
        gas_utility_col = c
        break

# Determine what column in the main df holds the electric utility name
electric_utility_col = None
for c in ['electric_utility', 'elec_utility', 'electricutility', 'electric_provider']:
    if c in df.columns:
        electric_utility_col = c
        break

# Compute percentage columns on df (stored so you can QA)
df['electric_incentive_pct'] = df.apply(
    lambda r: get_incentive_pct(r.get('measure_name'), r.get(electric_utility_col) if electric_utility_col else None, electric_pct_col),
    axis=1
)
df['natural_gas_incentive_pct'] = df.apply(
    lambda r: get_incentive_pct(r.get('measure_name'), r.get(gas_utility_col) if gas_utility_col else None, gas_pct_col),
    axis=1
)
df['nonutility_incentive_pct'] = df.apply(
    lambda r: get_incentive_pct(r.get('measure_name'), None, nonutility_pct_col),
    axis=1
)

# If the incentive percentages in your incentives file are expressed as whole percent (e.g., 50 for 50%),
# convert to decimals when a value > 1 is detected.
for pct_col in ['electric_incentive_pct', 'natural_gas_incentive_pct', 'nonutility_incentive_pct']:
    if df[pct_col].abs().max() > 1:
        df[pct_col] = df[pct_col] / 100.0

# Finally compute dollar incentives by multiplying the incremental cost per measure
df['electric_utility_incentive'] = df['measure_incremental_cost'] * df['electric_incentive_pct']
df['natural_gas_utility_incentive'] = df['measure_incremental_cost'] * df['natural_gas_incentive_pct']
df['nonutility_incentive'] = df['measure_incremental_cost'] * df['nonutility_incentive_pct']

# Quick QA print (small sample)
#print(df[['measure_name', electric_utility_col if electric_utility_col else 'electric_utility', gas_utility_col if gas_utility_col else 'gas_utility', 'electric_incentive_pct', 'electric_utility_incentive', 'natural_gas_incentive_pct', 'natural_gas_utility_incentive', 'nonutility_incentive_pct', 'nonutility_incentive']].tail(6))



In [ ]:
### Program Costs
# from 14_program costs
programs = pd.read_excel("input/14_Programs.xlsx")
clean_column_names(programs)

# will also need to adjust each utilities program costs allowing for non overlap in the fuel type groupings
# fuel type groups are needed for the tests but they all work together in that all added or all subtracted
# Merge df with programs on 'program' and 'fuel'
### Program Costs

# Robust mapping: match main df rows to programs by program, utility, and fuel,
# then compute non-measure program costs = measure_incremental_cost * non-incentive percent
# Detect possible column-name variants in the programs table
prog_program_col = None
for c in ['program', 'program_name', 'programs']:
    if c in programs.columns:
        prog_program_col = c
        break
prog_utility_col = None
for c in ['utility', 'utility_name', 'provider']:
    if c in programs.columns:
        prog_utility_col = c
        break
prog_fuel_col = None
for c in ['fuel', 'utility_fuel', 'fuel_type']:
    if c in programs.columns:
        prog_fuel_col = c
        break
# percent column (allow for dashes / underscores / long names)
pct_col = None
for c in programs.columns:
    low = c.lower()
    if 'non' in low and 'incent' in low and 'percent' in low:
        pct_col = c
        break
# fallback: look for 'non-incentive' then 'percent'
if pct_col is None:
    for c in programs.columns:
        low = c.lower()
        if 'non' in low and 'incent' in low:
            pct_col = c
            break
# If still None, try an exact expected name used historically
if pct_col is None and 'non-incentive_costs_as_a_percent_of_incentive_costs' in programs.columns:
    pct_col = 'non-incentive_costs_as_a_percent_of_incentive_costs'
# Defensive: if we still don't have a pct_col, set it to None and treat as zeros
if pct_col is None:
    print("Warning: could not find non-incentive percent column in programs; program-costs will be 0.")

# Helpers to normalize fuel names and detect match rows
def fuel_matches(val, desired):
    if pd.isna(val):
        return False
    v = str(val).lower()
    if desired == 'electric':
        return 'elect' in v or 'elec' in v
    if desired == 'natural_gas':
        return 'gas' in v and ('natural' in v or 'nat' in v) or v.strip() == 'gas'
    if desired == 'other':
        return 'other' in v or 'non' in v or 'oth' in v
    return desired in v

# Determine which columns in main df hold program, electric utility and gas utility
df_program_col = None
for c in ['program', 'program_name', 'programs']:
    if c in df.columns:
        df_program_col = c
        break
electric_utility_col = None
for c in ['electric_utility', 'elec_utility', 'electricutility', 'electric_provider']:
    if c in df.columns:
        electric_utility_col = c
        break
gas_utility_col = None
for c in ['gas_utility', 'natural_gas_utility', 'gasutility', 'gas_provider']:
    if c in df.columns:
        gas_utility_col = c
        break

# Function to lookup percent from programs table with fallbacks
def lookup_program_pct(prog_name, util_name, desired_fuel):
    # If pct_col is missing, return 0
    if pct_col is None:
        return 0.0
    # Try exact match on program, utility, and fuel
    candidates = programs
    if prog_program_col is not None and prog_name is not None:
        candidates = candidates[candidates[prog_program_col] == prog_name]
    if prog_utility_col is not None and util_name is not None:
        matched = candidates[candidates[prog_utility_col] == util_name]
        # keep candidates as matched if not empty for next step
        if not matched.empty:
            candidates = matched
    # Filter by fuel using fuzzy match
    # prefer exact fuel match rows
    fuel_mask = candidates[prog_fuel_col].apply(lambda x: fuel_matches(x, desired_fuel)) if prog_fuel_col in candidates.columns else pd.Series([False]*len(candidates), index=candidates.index)
    if fuel_mask.any():
        candidates = candidates[fuel_mask]
    # If we have any candidate rows, pick the first non-null pct_col
    if not candidates.empty:
        non_null = candidates[candidates[pct_col].notna()]
        if not non_null.empty:
            return float(non_null.iloc[0][pct_col])
        val = candidates.iloc[0].get(pct_col)
        return float(val) if pd.notna(val) else 0.0
    # Fallbacks: try any row with the fuel type across the whole table
    if prog_fuel_col in programs.columns:
        fuel_rows = programs[programs[prog_fuel_col].apply(lambda x: fuel_matches(x, desired_fuel))]
        if not fuel_rows.empty:
            non_null = fuel_rows[fuel_rows[pct_col].notna()]
            if not non_null.empty:
                return float(non_null.iloc[0][pct_col])
            val = fuel_rows.iloc[0].get(pct_col)
            return float(val) if pd.notna(val) else 0.0
    # Last resort: use first row's pct_col if present
    if pct_col in programs.columns and not programs.empty:
        val = programs.iloc[0].get(pct_col)
        return float(val) if pd.notna(val) else 0.0
    return 0.0

# Compute percent columns for each row in df and then multiply by incremental cost
def row_prog_costs(row):
    prog_name = row.get(df_program_col) if df_program_col else None
    elec_util = row.get(electric_utility_col) if electric_utility_col else None
    gas_util = row.get(gas_utility_col) if gas_utility_col else None
    # lookup percents
    elec_pct = lookup_program_pct(prog_name, elec_util, 'electric')
    gas_pct = lookup_program_pct(prog_name, gas_util, 'natural_gas')
    other_pct = lookup_program_pct(prog_name, None, 'other')
    # convert >1 to decimals
    for v in [elec_pct, gas_pct, other_pct]:
        pass
    if elec_pct > 1:
        elec_pct = elec_pct / 100.0
    if gas_pct > 1:
        gas_pct = gas_pct / 100.0
    if other_pct > 1:
        other_pct = other_pct / 100.0
    cost = row.get('measure_incremental_cost', 0.0)
    return pd.Series({
        'electric_utility_nonmeasure_program_cost': cost * elec_pct,
        'natural_gas_utility_nonmeasure_program_costs': cost * gas_pct,
        'nonutility_nonmeasure_program_costs': cost * other_pct
    })

# Apply to df (creates the three columns)
prog_costs_df = df.apply(row_prog_costs, axis=1)
df = pd.concat([df, prog_costs_df], axis=1)

# If downstream code expects these exact column names to exist (even if zeros), ensure they do
for col in ['electric_utility_nonmeasure_program_cost', 'natural_gas_utility_nonmeasure_program_costs', 'nonutility_nonmeasure_program_costs']:
    if col not in df.columns:
        df[col] = 0.0

# Excel QC: show a few rows
print(df[['measure_name', df_program_col if df_program_col else 'program', electric_utility_col if electric_utility_col else 'electric_utility', gas_utility_col if gas_utility_col else 'gas_utility', 'electric_utility_nonmeasure_program_cost', 'natural_gas_utility_nonmeasure_program_costs', 'nonutility_nonmeasure_program_costs']].head())

df

                                        measure_name program electric_utility  \
0  refrigerator_electricity_efficient_residential...  NLIRNC   test_utility_1   
1  refrigerator_electricity_efficient_residential...  NLIRNC   test_utility_1   
2  refrigerator_electricity_efficient_residential...  NLIRNC   test_utility_1   
3  refrigerator_electricity_efficient_residential...  NLIRNC   test_utility_1   
4  refrigerator_electricity_efficient_residential...  NLIRNC   test_utility_1   

      gas_utility  electric_utility_nonmeasure_program_cost  \
0  test_utility_2                                      65.0   
1  test_utility_2                                      65.0   
2  test_utility_2                                      65.0   
3  test_utility_2                                      65.0   
4  test_utility_2                                      45.0   

   natural_gas_utility_nonmeasure_program_costs  \
0                                         195.0   
1                               

                                        measure_name program electric_utility  \
0  refrigerator_electricity_efficient_residential...  NLIRNC   test_utility_1   
1  refrigerator_electricity_efficient_residential...  NLIRNC   test_utility_1   
2  refrigerator_electricity_efficient_residential...  NLIRNC   test_utility_1   
3  refrigerator_electricity_efficient_residential...  NLIRNC   test_utility_1   
4  refrigerator_electricity_efficient_residential...  NLIRNC   test_utility_1   

      gas_utility  electric_utility_nonmeasure_program_cost  \
0  test_utility_2                                      65.0   
1  test_utility_2                                      65.0   
2  test_utility_2                                      65.0   
3  test_utility_2                                      65.0   
4  test_utility_2                                      45.0   

   natural_gas_utility_nonmeasure_program_costs  \
0                                         195.0   
1                               

,measure_name,sector,program,electric_utility,gas_utility,market,baseline_condition,efficient_condition,building_type,electric_end_use,...,water_savings_value,electric_incentive_pct,natural_gas_incentive_pct,nonutility_incentive_pct,electric_utility_incentive,natural_gas_utility_incentive,nonutility_incentive,electric_utility_nonmeasure_program_cost,natural_gas_utility_nonmeasure_program_costs,nonutility_nonmeasure_program_costs
0,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,0.145013,0.5,0.75,0.5,325.0,487.5,325.0,65.0,195.0,195.0
1,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,...,0.145013,0.5,0.75,0.5,325.0,487.5,325.0,65.0,195.0,195.0
2,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,...,0.145013,0.5,0.75,0.5,325.0,487.5,325.0,65.0,195.0,195.0
3,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family,unknown,...,0.145013,0.5,0.75,0.5,325.0,487.5,325.0,65.0,195.0,195.0
4,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,0.145013,0.5,0.75,0.5,225.0,337.5,225.0,45.0,135.0,135.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,0.333208,0.0,0.50,0.5,0.0,38.5,38.5,23.1,23.1,23.1
300,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,RENO,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,0.333208,0.0,0.50,0.5,0.0,38.5,38.5,23.1,23.1,23.1
301,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,NC,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,0.333208,0.0,0.50,0.5,-0.0,-1025.0,-1025.0,-615.0,-615.0,-615.0
302,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,0.333208,0.0,0.50,0.5,-0.0,-1025.0,-1025.0,-615.0,-615.0,-615.0


In [ ]:
## Risk Discount
# going in the benefits section
# Risk Discount Factor to the incremental installed cost, any operation and maintenance costs, 
# and any deferred replacement credit (for early-retirement retrofits)

#Input from Global inputs sheet in workpapers


# Charactized as a benefit
risk_discount = 0.02
# each of these are already the total for the measure over EUL
df['risk_discount_value'] = (df['measure_incremental_cost'] +df['o_and_m_savings_value']+ df["deferred_replacement_credit_value"]) * risk_discount

""" For each efficiency measure, risk discount costs are calculated by applying the Risk Discount Factor to the incremental installed cost,
any operation and maintenance costs, and any deferred replacement credit (for early-retirement retrofits). 
The societal or total resource cost-effectiveness test costs are added to the societal or total resource benefits ."""

' For each efficiency measure, risk discount costs are calculated by applying the Risk Discount Factor to the incremental installed cost,\nany operation and maintenance costs, and any deferred replacement credit (for early-retirement retrofits). \nThe societal or total resource cost-effectiveness test costs are added to the societal or total resource benefits .'

In [ ]:
for ghg, suffix in [('carbon', '_usdpermmbtu_carbon'), ('n2o', '_usdpermmbtu_n2o'), ('ch4', '_usdpermmbtu_ch4')]:
    # Find columns for this gas
    ghg_cols = [col for col in avoided_costs.columns if ghg in col.lower()]
    avoided_costs_ghg = avoided_costs[ghg_cols].copy()
    # Remove suffix from column names
    avoided_costs_ghg.columns = [col.replace(suffix, '') for col in avoided_costs_ghg.columns]
    # Find common columns
    common = loadshapes.columns.intersection(avoided_costs_ghg.columns).intersection(line_losses.columns)
    # Calculate weights and losses
    weights = avoided_costs_ghg.loc[0, common].astype(float).fillna(0)
    losses = line_losses.loc[0, common].astype(float).fillna(0)
    adjusted_weights = weights * (1 - losses)
    period_value = loadshapes[common].fillna(0).dot(adjusted_weights)
    # Save to df
    df[f'electric_{ghg}_savings_value'] = period_value * df['measure_electric_energy_savings']
df
#value should be total value of a single measure 

,measure_name,sector,program,electric_utility,gas_utility,market,baseline_condition,efficient_condition,building_type,electric_end_use,...,electric_utility_incentive,natural_gas_utility_incentive,nonutility_incentive,electric_utility_nonmeasure_program_cost,natural_gas_utility_nonmeasure_program_costs,nonutility_nonmeasure_program_costs,risk_discount_value,electric_carbon_savings_value,electric_n2o_savings_value,electric_ch4_savings_value
0,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,325.0,487.5,325.0,65.0,195.0,195.0,14.706041,46.361688,0.102195,2.470273
1,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,...,325.0,487.5,325.0,65.0,195.0,195.0,14.706041,77.269480,0.170326,4.117122
2,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,...,325.0,487.5,325.0,65.0,195.0,195.0,14.706041,77.269480,0.170326,4.117122
3,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family,unknown,...,325.0,487.5,325.0,65.0,195.0,195.0,14.706041,47.526282,0.094513,2.597259
4,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,225.0,337.5,225.0,45.0,135.0,135.0,10.706041,30.552610,0.060758,1.669667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,0.0,38.5,38.5,23.1,23.1,23.1,5.460088,NaN,NaN,NaN
300,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,RENO,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,0.0,38.5,38.5,23.1,23.1,23.1,5.460088,NaN,NaN,NaN
301,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,NC,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,-0.0,-1025.0,-1025.0,-615.0,-615.0,-615.0,-37.079912,NaN,NaN,NaN
302,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,-0.0,-1025.0,-1025.0,-615.0,-615.0,-615.0,-37.079912,NaN,NaN,NaN


In [ ]:
# Externalities (GHG) calculations for all fuel types
# For each GHG type (carbon, n2o, ch4), process all fuel-type columns and compute lifetime NPV savings

# Fuel type configurations: (fuel_key, measure_column_name, sector_columns_map)
fuel_ghg_configs = [
    ('natural_gas', 'measure_natural_gas_savings', {
        'res': 'natural_gas',
        'com': 'natural_gas',
        'ind': 'natural_gas'
    }),
    ('fuel_oil', 'measure_fuel_oil_savings', {
        'res': 'res_fuel_oil',
        'com': 'com_fuel_oil',
        'ind': 'ind_fuel_oil'
    }),
    ('propane', 'measure_propane_savings', {
        'res': 'res_propane',
        'com': 'com_propane',
        'ind': 'ind_propane'
    }),
    ('diesel_transportation', 'measure_diesel_savings', {
        'res': 'diesel_transportation',
        'com': 'diesel_transportation',
        'ind': 'diesel_transportation'
    }),
    ('gasoline_transportation', 'measure_gasoline_savings', {
        'res': 'gasoline_transportation',
        'com': 'gasoline_transportation',
        'ind': 'gasoline_transportation'
    })
]

# Iterate over each GHG type (carbon, n2o, ch4)
for ghg, suffix in [('carbon', '_usdpermmbtu_carbon'), ('n2o', '_usdpermmbtu_n2o'), ('ch4', '_usdpermmbtu_ch4')]:
    # Find all columns for this GHG type in avoided_costs
    ghg_cols = [col for col in avoided_costs.columns if ghg in col.lower()]
    avoided_costs_ghg_fuel = avoided_costs[['year'] + ghg_cols].copy()
    # Remove suffix from column names to get fuel identifiers
    avoided_costs_ghg_fuel.columns = ['year'] + [col.replace(suffix, '') for col in ghg_cols]
    
    # Process each fuel type
    for fuel_type, measure_col, sector_columns in fuel_ghg_configs:
        output_col = f'{fuel_type}_{ghg}_savings_value'
        
        def calculate_fuel_ghg_savings(row):
            """Calculate lifetime NPV of GHG savings for a fuel type across sectors"""
            # Determine sector from the 'sector' column (case-insensitive)
            sector = None
            sector_lower = str(row.get('sector', '')).lower()
            
            if 'res' in sector_lower:
                sector = 'res'
            elif 'com' in sector_lower:
                sector = 'com'
            elif 'ind' in sector_lower:
                sector = 'ind'
            else:
                return 0  # No matching sector
            
            # Get the appropriate fuel column key for this sector
            fuel_col_key = sector_columns.get(sector)
            
            # Check if the fuel column exists in avoided_costs_ghg_fuel
            if fuel_col_key not in avoided_costs_ghg_fuel.columns:
                return 0
            
            # Check if measure savings column exists and has a non-zero value
            if measure_col not in row.index or pd.isna(row[measure_col]):
                return 0
            
            measure_savings = row[measure_col]
            if measure_savings == 0:
                return 0
            
            # Calculate lifetime NPV using the avoided cost series for this fuel and GHG
            try:
                avoided_cost_series = avoided_costs_ghg_fuel.set_index('year')[fuel_col_key]
                lifetime_npv = calculate_lifetime_savings(
                    measure_savings,
                    avoided_cost_series,
                    int(row['measure_life_(yrs)'])
                )
                return lifetime_npv
            except Exception as e:
                # Debug: print the error if needed
                # print(f"Error in {fuel_type}_{ghg}: {e}")
                return 0
        
        # Apply the calculation for this fuel+GHG combination
        df[output_col] = df.apply(calculate_fuel_ghg_savings, axis=1)

# Aggregate all fuel externalities into a single column (sum across all fuels and GHGs)
externality_cols = [col for col in df.columns if col.endswith(('_carbon_savings_value', '_n2o_savings_value', '_ch4_savings_value'))]
df["end_use_fuel_externalities"] = df[externality_cols].sum(axis=1) if externality_cols else 0

df

,measure_name,sector,program,electric_utility,gas_utility,market,baseline_condition,efficient_condition,building_type,electric_end_use,...,natural_gas_n2o_savings_value,fuel_oil_n2o_savings_value,propane_n2o_savings_value,diesel_transportation_n2o_savings_value,gasoline_transportation_n2o_savings_value,natural_gas_ch4_savings_value,fuel_oil_ch4_savings_value,propane_ch4_savings_value,diesel_transportation_ch4_savings_value,gasoline_transportation_ch4_savings_value
0,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,0.741863,4.451175,4.451175,0.445118,0.445118,0.275129,0.825386,0.825386,0.082539,0.082539
1,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,...,0.741863,4.451175,4.451175,0.445118,0.445118,0.275129,0.825386,0.825386,0.082539,0.082539
2,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,...,0.741863,4.451175,4.451175,0.445118,0.445118,0.275129,0.825386,0.825386,0.082539,0.082539
3,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family,unknown,...,0.741863,4.451175,4.451175,0.445118,0.445118,0.275129,0.825386,0.825386,0.082539,0.082539
4,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,0.741863,4.451175,4.451175,0.445118,0.445118,0.275129,0.825386,0.825386,0.082539,0.082539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,1.953999,11.723993,11.723993,1.172399,1.172399,0.803401,2.410204,2.410204,0.241020,0.241020
300,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,RENO,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,1.953999,11.723993,11.723993,1.172399,1.172399,0.803401,2.410204,2.410204,0.241020,0.241020
301,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,NC,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,1.953999,11.723993,11.723993,1.172399,1.172399,0.803401,2.410204,2.410204,0.241020,0.241020
302,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,1.953999,11.723993,11.723993,1.172399,1.172399,0.803401,2.410204,2.410204,0.241020,0.241020


In [ ]:
# Just a random adder
# In Penn this was DRIPE
#this is the fill in for MeasNonResource Tab 
# Nonresource benefits are matched by utility and NPV'd over measure lifetime with real discount rate

nonresource_benefits = pd.read_excel("input/16_measnonresource.xlsx")
clean_column_names(nonresource_benefits)

# Detect utility column in nonresource_benefits
nrb_utility_col = None
for c in ['utility', 'utility_name', 'provider']:
    if c in nonresource_benefits.columns:
        nrb_utility_col = c
        break

# Find the benefit value column (look for 'nonresource' and 'benefit')
nrb_value_col = None
for c in nonresource_benefits.columns:
    low = c.lower()
    if 'nonresource' in low and 'benefit' in low:
        nrb_value_col = c
        break
# Fallback: look for just 'benefit'
if nrb_value_col is None:
    for c in nonresource_benefits.columns:
        low = c.lower()
        if 'benefit' in low and 'other' in low:
            nrb_value_col = c
            break
# Fallback: try exact name
if nrb_value_col is None and 'other_nonresource_benefit' in nonresource_benefits.columns:
    nrb_value_col = 'other_nonresource_benefit'

# Detect utility columns in main df (electric_utility and gas_utility)
electric_utility_col = None
for c in ['electric_utility', 'elec_utility', 'electricutility', 'electric_provider']:
    if c in df.columns:
        electric_utility_col = c
        break
gas_utility_col = None
for c in ['gas_utility', 'natural_gas_utility', 'gasutility', 'gas_provider']:
    if c in df.columns:
        gas_utility_col = c
        break

# Helper function to lookup nonresource benefit for a given utility (with fallbacks)
def get_nonresource_benefit_value(utility_name):
    if nrb_value_col is None or nonresource_benefits.empty:
        return 0.0
    # Try exact match on utility if utility_col exists
    if nrb_utility_col is not None and utility_name is not None:
        match = nonresource_benefits[nonresource_benefits[nrb_utility_col] == utility_name]
        if not match.empty and pd.notna(match.iloc[0].get(nrb_value_col)):
            return float(match.iloc[0][nrb_value_col])
    # Fallback: use first row's value if available
    if not nonresource_benefits.empty:
        val = nonresource_benefits.iloc[0].get(nrb_value_col)
        return float(val) if pd.notna(val) else 0.0
    return 0.0

# Function to compute NPV of nonresource benefit across measure lifetime
def compute_npv_nonresource_benefit(row):
    """
    For each row in df:
    1. Look up nonresource benefit value for electric_utility
    2. Look up nonresource benefit value for gas_utility
    3. If they are different utilities, sum both; otherwise use single value
    4. Discount each year's benefit by (1 + 0.03)^year and sum across measure life
    """
    measure_life = int(row.get('measure_life_(yrs)', 1))
    elec_util = row.get(electric_utility_col) if electric_utility_col else None
    gas_util = row.get(gas_utility_col) if gas_utility_col else None
    
    # Lookup benefit values for each utility
    elec_benefit = get_nonresource_benefit_value(elec_util)
    gas_benefit = get_nonresource_benefit_value(gas_util)
    
    # If utilities differ, sum both benefits; otherwise just use one
    if elec_util != gas_util:
        total_annual_benefit = elec_benefit + gas_benefit
    else:
        # Same utility, use just one (avoid double-counting)
        total_annual_benefit = elec_benefit if elec_benefit != 0 else gas_benefit
    
    # NPV across measure lifetime with real_discount_rate = 0.03
    npv = sum(
        total_annual_benefit / ((1 + real_discount_rate) ** year)
        for year in range(1, measure_life + 1)
    )
    return npv

# Apply to df
df['other_nonresource_benefits'] = df.apply(compute_npv_nonresource_benefit, axis=1)

# QA: show sample
print("Nonresource Benefits (NPV over measure life):")
print(df[[electric_utility_col if electric_utility_col else 'electric_utility', 
          gas_utility_col if gas_utility_col else 'gas_utility', 
          'measure_life_(yrs)', 
          'other_nonresource_benefits']].head())

df

Nonresource Benefits (NPV over measure life):
  electric_utility     gas_utility  measure_life_(yrs)  \
0   test_utility_1  test_utility_2                  10   
1   test_utility_1  test_utility_2                  10   
2   test_utility_1  test_utility_2                  10   
3   test_utility_1  test_utility_2                  10   
4   test_utility_1  test_utility_2                  10   

   other_nonresource_benefits  
0                    8.746824  
1                    8.746824  
2                    8.746824  
3                    8.746824  
4                    8.746824  


Nonresource Benefits (NPV over measure life):
  electric_utility     gas_utility  measure_life_(yrs)  \
0   test_utility_1  test_utility_2                  10   
1   test_utility_1  test_utility_2                  10   
2   test_utility_1  test_utility_2                  10   
3   test_utility_1  test_utility_2                  10   
4   test_utility_1  test_utility_2                  10   

   other_nonresource_benefits  
0                    8.746824  
1                    8.746824  
2                    8.746824  
3                    8.746824  
4                    8.746824  


,measure_name,sector,program,electric_utility,gas_utility,market,baseline_condition,efficient_condition,building_type,electric_end_use,...,nonutility_incentive,electric_utility_nonmeasure_program_cost,natural_gas_utility_nonmeasure_program_costs,nonutility_nonmeasure_program_costs,risk_discount_value,electric_carbon_savings_value,electric_n2o_savings_value,electric_ch4_savings_value,end_use_fuel_externalities,other_nonresource_benefits
0,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,325.0,65.0,195.0,195.0,14.706041,46.361688,0.102195,2.470273,0,8.746824
1,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,...,325.0,65.0,195.0,195.0,14.706041,77.269480,0.170326,4.117122,0,8.746824
2,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,...,325.0,65.0,195.0,195.0,14.706041,77.269480,0.170326,4.117122,0,8.746824
3,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family,unknown,...,325.0,65.0,195.0,195.0,14.706041,47.526282,0.094513,2.597259,0,8.746824
4,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,225.0,45.0,135.0,135.0,10.706041,30.552610,0.060758,1.669667,0,8.746824
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,38.5,23.1,23.1,23.1,5.460088,NaN,NaN,NaN,0,39.200883
300,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,RENO,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,38.5,23.1,23.1,23.1,5.460088,NaN,NaN,NaN,0,39.200883
301,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,NC,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,-1025.0,-615.0,-615.0,-615.0,-37.079912,NaN,NaN,NaN,0,39.200883
302,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,-1025.0,-615.0,-615.0,-615.0,-37.079912,NaN,NaN,NaN,0,39.200883


In [ ]:
# Retail rates: map sector and fuel to correct retail price columns and compute NPV of customer bill savings
# retail_rates contains columns like:\n
# electricity_retail_prices_res_usdperkwh, electricity_retail_prices_commercial_usdperkwh, electricity_retail_prices_industrial_usdperkwh,\n
# natural_gas_retail_prices_res_usdpermmbtu, natural_gas_retail_prices_commercial_usdpermmbtu, natural_gas_retail_prices_industrial_usdpermmbtu,\n
# heating_oil_retail_prices_res_usdpermmbtu, heating_oil_retail_prices_commercial_usdpermmbtu, heating_oil_retail_prices_industrial_usdpermmbtu, and year

# Read retail rates (adjust path if you have a dedicated retail rates file)
retail_rates = pd.read_excel('input/11_Avoided_Cost.xlsx')  # change to 'input/17_retail_rates.xlsx' if available
clean_column_names(retail_rates)
# keep only retail price related columns and year
cols = [c for c in retail_rates.columns if ('retail_price' in c.lower()) or ('retail_prices' in c.lower()) or ('year' in c.lower())]
retail_rates = retail_rates[cols].copy()

# Helper maps for expected column patterns
prefix_map = {
    'electric': 'electricity_retail_prices',
    'natural_gas': 'natural_gas_retail_prices',
    'heating_oil': 'heating_oil_retail_prices'
}
# unit suffixes seen in the file (electric kWh, fuels per MMBtu)
suffix_map = { 'electric': '_usdperkwh', 'natural_gas': '_usdpermmbtu', 'heating_oil': '_usdpermmbtu' }

# Robust column finder: try exact constructed name then fallback to contains-based search
def find_retail_col(fuel_key, sector_key):
    # exact name first
    pref = prefix_map.get(fuel_key)
    suf = suffix_map.get(fuel_key)
    if pref is None or suf is None:
        return None
    exact = f'{pref}_{sector_key}{suf}'
    if exact in retail_rates.columns:
        return exact
    # try variants: some files use full 'commercial' others 'com' or 'res' etc. search by parts
    for col in retail_rates.columns:
        low = col.lower()
        if pref.replace('_', '') in low and sector_key in low:
            return col
    # fallback: any column that contains the prefix
    for col in retail_rates.columns:
        if pref.replace('_', '') in col.lower():
            return col
    return None

def sector_key_from_sector_text(sector_text):
    s = str(sector_text).lower() if sector_text is not None else ''
    if 'res' in s or 'resident' in s or 'house' in s:
        return 'res'
    if 'com' in s or 'commercial' in s:
        return 'commercial'
    if 'ind' in s or 'industrial' in s:
        return 'industrial'
    # default to error if unknown
    return 'error'

# Build series cache to avoid repeated set_index operations
_retail_series_cache = {}
def get_retail_series(fuel_key, sector_key):
    cache_key = (fuel_key, sector_key)
    if cache_key in _retail_series_cache:
        return _retail_series_cache[cache_key]
    col = find_retail_col(fuel_key, sector_key)
    if col is None:
        _retail_series_cache[cache_key] = None
        return None
    try:
        series = retail_rates.set_index('year')[col]
    except Exception:
        _retail_series_cache[cache_key] = None
        return None
    _retail_series_cache[cache_key] = series
    return series

# For each row, compute lifetime (NPV-like) customer bill savings for electricity, natural gas, heating oil
def compute_retail_savings_row(row):
    life = int(row.get('measure_life_(yrs)', 1)) if pd.notna(row.get('measure_life_(yrs)')) else 1
    sector_txt = row.get('sector', '')
    sector_k = sector_key_from_sector_text(sector_txt)
    out = {'electric_customer_bill_savings': 0.0, 'natural_gas_customer_bill_savings': 0.0, 'heating_oil_customer_bill_savings': 0.0}
    # Electricity
    try:
        elec_sav = row.get('measure_electric_energy_savings', 0)
    except Exception:
        elec_sav = 0
    if pd.notna(elec_sav) and elec_sav != 0:
        series = get_retail_series('electric', sector_k)
        if series is not None:
            out['electric_customer_bill_savings'] = calculate_lifetime_savings(elec_sav, series, life)
    # Natural gas
    try:
        gas_sav = row.get('measure_natural_gas_savings', 0)
    except Exception:
        gas_sav = 0
    if pd.notna(gas_sav) and gas_sav != 0:
        series = get_retail_series('natural_gas', sector_k)
        if series is not None:
            out['natural_gas_customer_bill_savings'] = calculate_lifetime_savings(gas_sav, series, life)
    # Heating oil
    try:
        oil_sav = row.get('measure_fuel_oil_savings', 0)
    except Exception:
        oil_sav = 0
    if pd.notna(oil_sav) and oil_sav != 0:
        series = get_retail_series('heating_oil', sector_k)
        if series is not None:
            out['heating_oil_customer_bill_savings'] = calculate_lifetime_savings(oil_sav, series, life)
    return pd.Series(out)

# Apply and attach to df
retail_savings_df = df.apply(compute_retail_savings_row, axis=1)
df = pd.concat([df, retail_savings_df], axis=1)

# Quick QA
print('Retail customer bill savings sample:')
print(df[['sector','measure_life_(yrs)','measure_electric_energy_savings','electric_customer_bill_savings','measure_natural_gas_savings','natural_gas_customer_bill_savings','measure_fuel_oil_savings','heating_oil_customer_bill_savings']].head())

# If downstream code expects 'electric_customer_bill_savings' to exist, ensure column exists (even if all zeros)
for col in ['electric_customer_bill_savings','natural_gas_customer_bill_savings','heating_oil_customer_bill_savings']:
    if col not in df.columns:
        df[col] = 0.0
df

Retail customer bill savings sample:
        sector  measure_life_(yrs)  measure_electric_energy_savings  \
0  residential                  10                       700.000000   
1  residential                  10                      1166.666667   
2  residential                  10                      1166.666667   
3  residential                  10                       700.000000   
4  residential                  10                       450.000000   

   electric_customer_bill_savings  measure_natural_gas_savings  \
0                     1512.000000                           10   
1                     2520.000001                           10   
2                     2520.000001                           10   
3                     1512.000000                           10   
4                      972.000000                           10   

   natural_gas_customer_bill_savings  measure_fuel_oil_savings  \
0                             1082.2                        10   
1      

Retail customer bill savings sample:
        sector  measure_life_(yrs)  measure_electric_energy_savings  \
0  residential                  10                       700.000000   
1  residential                  10                      1166.666667   
2  residential                  10                      1166.666667   
3  residential                  10                       700.000000   
4  residential                  10                       450.000000   

   electric_customer_bill_savings  measure_natural_gas_savings  \
0                     1512.000000                           10   
1                     2520.000001                           10   
2                     2520.000001                           10   
3                     1512.000000                           10   
4                      972.000000                           10   

   natural_gas_customer_bill_savings  measure_fuel_oil_savings  \
0                             1082.2                        10   
1      

,measure_name,sector,program,electric_utility,gas_utility,market,baseline_condition,efficient_condition,building_type,electric_end_use,...,nonutility_nonmeasure_program_costs,risk_discount_value,electric_carbon_savings_value,electric_n2o_savings_value,electric_ch4_savings_value,end_use_fuel_externalities,other_nonresource_benefits,electric_customer_bill_savings,natural_gas_customer_bill_savings,heating_oil_customer_bill_savings
0,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,195.0,14.706041,46.361688,0.102195,2.470273,0,8.746824,1512.000000,1082.2,2674.8
1,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,...,195.0,14.706041,77.269480,0.170326,4.117122,0,8.746824,2520.000001,1082.2,2674.8
2,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,...,195.0,14.706041,77.269480,0.170326,4.117122,0,8.746824,2520.000001,1082.2,2674.8
3,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family,unknown,...,195.0,14.706041,47.526282,0.094513,2.597259,0,8.746824,1512.000000,1082.2,2674.8
4,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,135.0,10.706041,30.552610,0.060758,1.669667,0,8.746824,972.000000,1082.2,2674.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,23.1,5.460088,NaN,NaN,NaN,0,39.200883,0.000000,3505.4,8704.5
300,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,RENO,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,23.1,5.460088,NaN,NaN,NaN,0,39.200883,0.000000,3505.4,8704.5
301,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,NC,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,-615.0,-37.079912,NaN,NaN,NaN,0,39.200883,705.000000,3505.4,8704.5
302,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,-615.0,-37.079912,NaN,NaN,NaN,0,39.200883,705.000000,3505.4,8704.5


In [ ]:
# All variables All units are in Dollars even savings in real dollars of the starting year

# measure_incremental_cost
# electric_utility_incentive
# electric_utility_nonmeasure_program_cost # need to have these separate as the electric and natural has utilities may be different based on the location of customers
# electric_customer_bill_savings # this is  retail cost of each energy source times energy used
# electric_energy_savings                  # all savings could also be increased useage which would be a negative value instead
# electric_demand_savings                  #- How does T&D and capacity factor in here? (loadshapes?)
# natural_gas_savings
# natural_gas_utility_incentive          # (incentive payment)
# natural_gas_utility_nonmeasure_program_costs
# other_fuel_savings
# nonunitility_incentive
# nonutility_nonmeasure_program_costs
# water_savings
# o_and_m_savings                          #could also be a cost in which case make negitive
# other_nonresource_benefits 
# deferred_replacement_credit_savings   #???????
# risk_discount
# electric_externalities                  # CO2, NOx, CH4
# end_use_fuel_externalities             # Included Natural Gas and Other Fuels



In [ ]:
# this is where we consolidate all the df to have all years
#then we split it down to just the first year for the cost tests

This is the equations for the 4 cost tests plus the three version of Utility cost test

In [ ]:
# electric utility only cost test
df["EUCT_cost"] = df["electric_utility_nonmeasure_program_cost"] + df["electric_utility_incentive"]

df["EUCT_benefit"] = df["electric_energy_savings_value"] + df["electric_demand_savings_value"] 

df["EUCT_BCR"] = df["EUCT_benefit"] / df["EUCT_cost"]

In [ ]:
# Natural Gas utility only cost test
df["GUCT_cost"] = df["natural_gas_utility_nonmeasure_program_costs"] + df["natural_gas_utility_incentive"]

df["GUCT_benefit"] = df["natural_gas_savings_value"] 

df["GUCT_BCR"] = df["GUCT_benefit"] / df["GUCT_cost"]

In [ ]:
# Electric and Natural Gas utility cost test
df["EGUCT_cost"] = df["natural_gas_utility_nonmeasure_program_costs"] + df["natural_gas_utility_incentive"] + df["electric_utility_nonmeasure_program_cost"] + df["electric_utility_incentive"]

df["EGUCT_benefit"] = df["natural_gas_savings_value"] + df["electric_energy_savings_value"] + df["electric_demand_savings_value"] 

df["EGUCT_BCR"] = df["EGUCT_benefit"] / df["EGUCT_cost"]

In [ ]:
# TRC
# incremental cost for ER is the full cost of the install then when the Deferred replacemnet credit happenes that is the balancing out 
# this way the retrofit is properly priced
df["TRC_cost"] = df["measure_incremental_cost"] + df["electric_utility_nonmeasure_program_cost"] + df["natural_gas_utility_nonmeasure_program_costs"] + df["nonutility_incentive"] + df["nonutility_nonmeasure_program_costs"]

df["TRC_benefit"] = df["electric_energy_savings_value"] + df["electric_demand_savings_value"] + df["natural_gas_savings_value"] + df["other_fuel_savings_value"] + df["water_savings_value"] + df["o_and_m_savings_value"] + df["other_nonresource_benefits"] + df["deferred_replacement_credit_value"] + df["risk_discount_value"]
df["TRC_BCR"] = df["TRC_benefit"] / df["TRC_cost"]

In [ ]:
# SCT
df["SCT_cost"] = df["measure_incremental_cost"] + df["electric_utility_nonmeasure_program_cost"] + df["natural_gas_utility_nonmeasure_program_costs"] + df["nonutility_incentive"] + df["nonutility_nonmeasure_program_costs"]

df["SCT_benefit"] = df["electric_energy_savings_value"] + df["electric_demand_savings_value"] + df["natural_gas_savings_value"] + df["other_fuel_savings_value"] + df["water_savings_value"] + df["o_and_m_savings_value"] + df["other_nonresource_benefits"] + df["deferred_replacement_credit_value"] + df["risk_discount_value"] + df["electric_carbon_savings_value"] + df["electric_n2o_savings_value"] + df["electric_ch4_savings_value"] + df["end_use_fuel_externalities"]
df["SCT_BCR"] = df["SCT_benefit"] / df["SCT_cost"]

In [ ]:
# RIM
# Why is this just electric bill savings not the other fuels?

 # RIM
df["RIM_cost"] = df["electric_utility_incentive"] + df["electric_utility_nonmeasure_program_cost"] + df["electric_customer_bill_savings"] + df["natural_gas_utility_incentive"] + df["natural_gas_utility_nonmeasure_program_costs"]

df["RIM_benefit"] = df["electric_energy_savings_value"] + df["electric_demand_savings_value"] + df["natural_gas_savings_value"] + df["other_fuel_savings_value"] + df["nonutility_incentive"] + df["nonutility_nonmeasure_program_costs"]
df["RIM_BCR"] = df["RIM_benefit"] / df["RIM_cost"]

In [ ]:
# PCT
# double check the electric customer cost thing I think we need all fuels bill savings or at least an option to change it
df["PCT_cost"] = df["measure_incremental_cost"]
# currently deferred replacement value is a benefit
df["PCT_benefit"] = df["electric_utility_incentive"] + df["electric_customer_bill_savings"] + df["electric_energy_savings_value"] + df["electric_demand_savings_value"] + df["natural_gas_savings_value"] + df["natural_gas_utility_incentive"] + df["other_fuel_savings_value"] + df["nonutility_incentive"] + df["water_savings_value"] + df["o_and_m_savings_value"] + df["other_nonresource_benefits"] + df["deferred_replacement_credit_value"]
df["PCT_BCR"] = df["PCT_benefit"] / df["PCT_cost"]

In [ ]:
df

,measure_name,sector,program,electric_utility,gas_utility,market,baseline_condition,efficient_condition,building_type,electric_end_use,...,TRC_BCR,SCT_cost,SCT_benefit,SCT_BCR,RIM_cost,RIM_benefit,RIM_BCR,PCT_cost,PCT_benefit,PCT_BCR
0,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,4.143571,1430.0,5974.240073,4.177790,2584.500000,6336.406010,2.451695,650,8560.099876,13.169384
1,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family,unknown,...,4.304174,1430.0,6236.525390,4.361207,3592.500001,6566.068557,1.827716,650,9797.762424,15.073481
2,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,multi_family_li,unknown,...,4.304174,1430.0,6236.525390,4.361207,3592.500001,6566.068557,1.827716,650,9797.762424,15.073481
3,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,RET_ER,refrigerator_electricity_existing_residential,refrigerator_electricity_efficient_residential,single_family,unknown,...,4.143571,1430.0,5975.523971,4.178688,2584.500000,6336.406010,2.451695,650,8560.099876,13.169384
4,refrigerator_electricity_efficient_residential...,residential,NLIRNC,test_utility_1,test_utility_2,NC,refrigerator_electricity_baseline_residential,refrigerator_electricity_efficient_residential,single_family_li,unknown,...,5.856841,990.0,5830.555444,5.889450,1714.500000,6053.372503,3.530693,450,7547.066369,16.771259
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,74.218172,184.8,NaN,NaN,84.700000,13536.119510,159.812509,77,13787.058014,179.052701
300,insulation_natural_gas_efficient_residential_w...,residential,NLIRRepl,none,test_utility_3,RENO,whole_home_natural_gas_baseline_residential,insulation_natural_gas_efficient_residential,single_family,unknown,...,74.218172,184.8,NaN,NaN,84.700000,13536.119510,159.812509,77,13787.058014,179.052701
301,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,NC,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,-2.795412,-4920.0,NaN,NaN,-1550.000000,11914.969669,-7.687077,-2050,12445.508173,-6.070980
302,HERS_electricity_efficient_residential_whole_h...,residential,NLIRRepl,none,test_utility_3,ROB,whole_home_natural_gas_baseline_residential,HERS_electricity_efficient_residential,single_family,unknown,...,-2.795412,-4920.0,NaN,NaN,-1550.000000,11914.969669,-7.687077,-2050,12445.508173,-6.070980


In [ ]:
df.to_csv("output/measure_costs_benefits.csv", index=False)

This is the output needed for mike to join to the measure database
It is boolen of each cost test pass or fail and than the PCT ratio (key parts)
Will provide everything to Mike

This is an example output needed for deliverable MeasScrn & MeasCostEff

In [ ]:
# Measure Name
# Primary Fuel
# 'Program'
# "Measure ID1"
# Sector
# "Building Type/Segment"	
# Primary Fuel 
# End Use	
# "Market(e.g., RET, NC, RENO, REPL)"	
# # Benefits and costs for all 4 tests totaled # done for only the first year of the study
# "Total Resource Benefits" = TRC_benefit
# "Total Resource Costs"  = TRC_cost
# "Total Resource Net Benefits" = TRC_benefit - TRC_cost
# "Total Resource BCR" = TRC_benefit / TRC_cost

In [ ]:
# Column names of EMeasure Name
# Primary Fuel
# Include in Calc's
# Measure ID3
# "Measure ID1
# Sector"	
# "Building Type/Segment"	
# Primary Fuel 
# End Use	
# "Market(e.g., RET, NC, RENO, REPL)"
# First Install Year
# Last Install Year
# "Incremental Installed Cost($)"
# "Retrofit deferral credit($)"
# "O&M ($)"
# "Fossil Fuel($)"
# "Fossil Fuel Externalites($)"
# "Risk Mitigation($)"
# "Measure lifetime(years)"
# "Levelized Annual Electric Energy savings(MWh/yr)"
# "Levelized Annual Summer Peak demand savings (kW-yr)"
# "Levelized Annual Winter Peak demand  savings (kW-yr)"
# "Generating Capacity Value of Peak  Demand savings($/kW-yr)"
# "T&D Capacity Value of Peak Demand savings($/kW-yr)"
# "System value of Electric Energy savings($/kWh)"
# "Total Value of Electricity Savings($)"	
# "Environ-mental Externalities($)"	
# "Fossil Fuel($)"	
# "Fossil Fuel Externalities($)"	
# "Water($)"	
# "Total Value of Total Resource Benefits($)"
# "Net Total Resource Benefits($)"
# Total Resource Benefit/ Cost Ratio
# "Net Levelized Cost per kWh ($/kWh)"
# "Net Cost Per Summer Peak kW-yr ($/kW-yr)"
# "Net Cost Per Winter Peak kW-yr ($/kW-yr)"



In [ ]:
# #Archive


# carbon_cols = [col for col in avoided_costs.columns if 'carbon' in col.lower()]
# avoided_costs_carbon = avoided_costs[carbon_cols]
# avoided_costs_carbon
# #now edit the carbon avoided columns to match the other tables so we can use the function
# avoided_costs_carbon.columns = [col.replace('_usdpermmbtu_carbon', '') for col in avoided_costs_carbon.columns]
# # Electric Energy Savings Calculation with line-loss adjustment
# # Find columns common to all three tables (avoided_costs, loadshapes, line_losses)
# common = loadshapes.columns.intersection(avoided_costs_carbon.columns).intersection(line_losses.columns)
# # Choose the appropriate row from avoided_costs and line_losses (adjust index/selection if needed)
# weights = avoided_costs_carbon.loc[0, common].astype(float).fillna(0)

# #will need to adjust sector selection based on measure sector
# losses = line_losses.loc[line_losses['sectors'] == 'res', common].iloc[0].astype(float).fillna(0)

# # Adjust weights by (1 - line_loss) so each period is: avoided_cost * (1 - line_loss)
# adjusted_weights = weights * (1 - losses)
# # Compute the vectorized sum across matching columns: for each row in loadshapes sum(loadshape * adjusted_weight)
# period_value = loadshapes[common].fillna(0).dot(adjusted_weights)
# # Multiply by the measure-level energy savings (broadcasting). If df and loadshapes indices differ
# # this will align by index; adjust broadcast strategy if you need a scalar multiplication instead.
# df['electric_carbon_savings_value'] = period_value * df['measure_electric_energy_savings']
# df     # CO2, NOx, CH4
# ###########
# n2o_cols = [col for col in avoided_costs.columns if 'n2o' in col.lower()]
# avoided_costs_n2o = avoided_costs[n2o_cols]
# avoided_costs_n2o
# #now edit the carbon avoided columns to match the other tables so we can use the function
# avoided_costs_n2o.columns = [col.replace('_usdpermmbtu_n2o', '') for col in avoided_costs_n2o.columns]
# # Electric Energy Savings Calculation with line-loss adjustment
# # Find columns common to all three tables (avoided_costs, loadshapes, line_losses)
# common = loadshapes.columns.intersection(avoided_costs_n2o.columns).intersection(line_losses.columns)
# # Choose the appropriate row from avoided_costs and line_losses (adjust index/selection if needed)
# weights = avoided_costs_n2o.loc[0, common].astype(float).fillna(0)

# losses = line_losses.loc[line_losses['sectors'] == 'res', common].iloc[0].astype(float).fillna(0)

# # Adjust weights by (1 - line_loss) so each period is: avoided_cost * (1 - line_loss)
# adjusted_weights = weights * (1 - losses)
# # Compute the vectorized sum across matching columns: for each row in loadshapes sum(loadshape * adjusted_weight)
# period_value = loadshapes[common].fillna(0).dot(adjusted_weights)
# # Multiply by the measure-level energy savings (broadcasting). If df and loadshapes indices differ
# # this will align by index; adjust broadcast strategy if you need a scalar multiplication instead.
# df['electric_n2o_savings_value'] = period_value * df['measure_electric_energy_savings']
# df     # CO2, NOx, CH4
# ###########
# ch4_cols = [col for col in avoided_costs.columns if '_ch4' in col.lower()]
# avoided_costs_ch4 = avoided_costs[ch4_cols]
# avoided_costs_ch4
# #now edit the carbon avoided columns to match the other tables so we can use the function
# avoided_costs_ch4.columns = [col.replace('_usdpermmbtu_ch4', '') for col in avoided_costs_ch4.columns]
# # Electric Energy Savings Calculation with line-loss adjustment
# # Find columns common to all three tables (avoided_costs, loadshapes, line_losses)
# common = loadshapes.columns.intersection(avoided_costs_ch4.columns).intersection(line_losses.columns)
# # Choose the appropriate row from avoided_costs and line_losses (adjust index/selection if needed)
# weights = avoided_costs_ch4.loc[0, common].astype(float).fillna(0)

# losses = line_losses.loc[line_losses['sectors'] == 'res', common].iloc[0].astype(float).fillna(0)

# # Adjust weights by (1 - line_loss) so each period is: avoided_cost * (1 - line_loss)
# adjusted_weights = weights * (1 - losses)
# # Compute the vectorized sum across matching columns: for each row in loadshapes sum(loadshape * adjusted_weight)
# period_value = loadshapes[common].fillna(0).dot(adjusted_weights)
# # Multiply by the measure-level energy savings (broadcasting). If df and loadshapes indices differ
# # this will align by index; adjust broadcast strategy if you need a scalar multiplication instead.
# df['electric_ch4_savings_value'] = period_value * df['measure_electric_energy_savings']
# df     # CO2, NOx, CH4